# ScienceQA Vision Challenge — **Inference Notebook**

Loads the LoRA adapter saved by `train.ipynb`, runs **log-likelihood scoring** on `test.csv`, and writes a competition-ready `submission.csv`.

Includes optional **TTA over choice permutations** — averages logits over `K` random orderings of the choice list to remove positional bias from the prediction. Empirically this is worth ~1–3 points on small VLMs.

> **Set `ADAPTER_DIR` to the path produced by `train.ipynb` (it's `outputs_train/adapter` by default).**


## 0. Install dependencies

If running offline (no internet at submit time) make sure these are installed in the environment ahead of time.

In [ ]:
# Comment out if your offline environment already has these
%pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes==0.44.1 accelerate==1.0.1 pillow

## 1. Imports + paths

In [ ]:
import os, json, math, random
from pathlib import Path
from typing import List, Optional, Dict, Any

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === EDIT THESE PATHS ===========================================
DATA_DIR    = Path("data")                        # contains test.csv + images/test/*.png
ADAPTER_DIR = Path("outputs_train/adapter")       # produced by train.ipynb
OUT_CSV     = Path("submission.csv")
# =================================================================

# Inference knobs
MODEL_ID         = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE         = 384
USE_4BIT         = False                          # set True on T4
TTA_PERMUTATIONS = 4                              # 1 = no TTA. 4–8 is a sweet spot.
EVAL_BATCH       = 8
MAX_CONTEXT_CHARS = 1200

assert (DATA_DIR / "test.csv").exists(), f"test.csv missing under {DATA_DIR}"
assert ADAPTER_DIR.exists(), f"Adapter dir not found: {ADAPTER_DIR}"
print("device:", device)
print("adapter:", ADAPTER_DIR.resolve())
if (ADAPTER_DIR / "train_meta.json").exists():
    print("train_meta:", json.loads((ADAPTER_DIR / 'train_meta.json').read_text()))

## 2. Load test data

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(json.loads)
print("test rows:", len(test_df))
test_df.head(2)

## 3. Load processor + base model + LoRA adapter

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"   # critical for next-token logit reads

model_kwargs: Dict[str, Any] = dict(low_cpu_mem_usage=True)
if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["dtype"] = torch.bfloat16
    model_kwargs["device_map"] = "auto" if torch.cuda.is_available() else None

base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
if not torch.cuda.is_available(): base.to(device)

model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
model.config.use_cache = True   # OK at inference
print("model + adapter loaded:", type(model).__name__)

## 4. Prompt + scoring (must match train.ipynb)

In [ ]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def _truncate(s: str, n: int) -> str:
    s = str(s).strip()
    return s if len(s) <= n else s[: n-1] + "…"

def build_user_text(row, choices: List[str], max_ctx=MAX_CONTEXT_CHARS,
                    use_lecture=True, use_hint=True) -> str:
    parts = []
    if use_lecture:
        lec = row.get("lecture", None)
        if isinstance(lec, str) and lec.strip(): parts.append(_truncate(lec, max_ctx))
    if use_hint:
        hint = row.get("hint", None)
        if isinstance(hint, str) and hint.strip(): parts.append(_truncate(hint, max_ctx//2))
    context = "\n".join(parts)
    choice_lines = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    text = ""
    if context: text += f"Context:\n{context}\n\n"
    text += f"Question: {row['question']}\n\nChoices:\n{choice_lines}\n\nAnswer with a single letter only."
    return text


def build_messages(row, choices):
    return [{"role":"user","content":[{"type":"image"},{"type":"text","text":build_user_text(row,choices)}]}]


def _letter_token_id(letter: str) -> int:
    ids = processor.tokenizer(letter, add_special_tokens=False).input_ids
    if len(ids) != 1:
        ids = processor.tokenizer(" " + letter, add_special_tokens=False).input_ids
    return ids[-1]

LETTER_IDS = {L: _letter_token_id(L) for L in CHOICE_LETTERS}
print("letter token ids:", LETTER_IDS)

In [ ]:
def load_image(rel_path):
    img = Image.open(DATA_DIR / rel_path).convert("RGB")
    w, h = img.size
    s = IMG_SIZE / max(w, h)
    if s < 1.0:
        img = img.resize((max(1,int(w*s)), max(1,int(h*s))), Image.BICUBIC)
    return img


@torch.inference_mode()
def score_logits(rows, images, choices_list):
    """Return per-row logits over the candidate letters (variable length)."""
    prompts = [processor.apply_chat_template(build_messages(r, ch), add_generation_prompt=True)
               for r, ch in zip(rows, choices_list)]
    inputs = processor(text=prompts, images=[[im] for im in images],
                       return_tensors="pt", padding=True)
    inputs = {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in inputs.items()}
    out = model(**inputs)
    last = out.logits[:, -1, :]   # (B, V); left-padded → -1 is the next-token slot
    per_row = []
    for i, ch in enumerate(choices_list):
        ids = [LETTER_IDS[CHOICE_LETTERS[k]] for k in range(len(ch))]
        per_row.append(last[i, ids].float().cpu().numpy())
    return per_row

## 5. Run inference (with optional TTA)

For each test row we evaluate the model under `TTA_PERMUTATIONS` random orderings of the choice list, project the logits back to the original index space, and average. Set `TTA_PERMUTATIONS = 1` to disable TTA.

In [ ]:
def tta_predict(test_rows, batch_size=EVAL_BATCH, n_perm=TTA_PERMUTATIONS, seed=SEED):
    """Returns list of predicted original-index ints, same length/order as test_rows."""
    n = len(test_rows)
    summed = [None] * n        # accumulated logits in ORIGINAL index space, per row
    rng = np.random.RandomState(seed)

    # pre-load all images once
    images = [load_image(r["image_path"]) for r in test_rows]

    for p in range(n_perm):
        # build per-row permutation
        perms = []
        for r in test_rows:
            k = len(r["choices"])
            perms.append(np.arange(k) if p == 0 else rng.permutation(k))

        for start in range(0, n, batch_size):
            sl = slice(start, min(start+batch_size, n))
            rows_b = test_rows[sl]
            imgs_b = images[sl]
            perms_b = perms[sl]
            shuffled_choices = [[r["choices"][i] for i in perm] for r, perm in zip(rows_b, perms_b)]
            logits_list = score_logits(rows_b, imgs_b, shuffled_choices)
            for i, (logits, perm) in enumerate(zip(logits_list, perms_b)):
                global_i = start + i
                # logits[j] = logit for "the j-th choice in shuffled order"
                # corresponds to original index perm[j]
                if summed[global_i] is None:
                    summed[global_i] = np.zeros(len(perm), dtype=np.float64)
                for j, orig_idx in enumerate(perm):
                    summed[global_i][orig_idx] += float(logits[j])
        print(f"  TTA pass {p+1}/{n_perm} done")

    preds = [int(np.argmax(s)) for s in summed]
    return preds, summed


# Convert the test_df to row-dicts so the function is fast/independent
test_rows = [{
    "id"        : r["id"],
    "image_path": r["image_path"],
    "question"  : r["question"],
    "choices"   : r["choices"],
    "lecture"   : r.get("lecture"),
    "hint"      : r.get("hint"),
} for _, r in test_df.iterrows()]

print(f"Predicting on {len(test_rows)} test rows with {TTA_PERMUTATIONS} TTA passes …")
preds, raw_logits = tta_predict(test_rows)
print("done. preds:", len(preds))

## 6. Build `submission.csv` with sanity checks

In [ ]:
sub = pd.DataFrame({"id": [r["id"] for r in test_rows], "answer": preds})

# Sanity checks
assert list(sub.columns) == ["id", "answer"], "columns must be exactly ['id','answer']"
assert len(sub) == len(test_df), "row count mismatch"
assert set(sub["id"]) == set(test_df["id"]), "id mismatch with test.csv"
assert sub["answer"].dtype.kind in ("i", "u"), "answer must be integer"
# Each predicted index must be a legal choice index for that row
for r, p in zip(test_rows, preds):
    assert 0 <= p < len(r["choices"]), f"id {r['id']} produced illegal index {p} for {len(r['choices'])} choices"

sub.to_csv(OUT_CSV, index=False)
print(f"✅ wrote {OUT_CSV}  ({len(sub)} rows)")
print("\nAnswer distribution:")
print(sub["answer"].value_counts().sort_index())
print("\nFirst 5 rows:")
print(sub.head())

## 7. (Optional) Cross-check vs zero-shot

If you also kept a zero-shot prediction CSV around, compare the two to make sure fine-tuning actually moved predictions (a healthy fine-tune typically changes 30–50 % of test predictions).

In [ ]:
# Example skeleton — uncomment if you saved a zero-shot submission earlier
# zs = pd.read_csv("submission_zeroshot.csv")
# diff = (zs.set_index('id')['answer'] != sub.set_index('id')['answer']).mean()
# print(f"fraction of predictions changed by fine-tuning: {diff*100:.1f}%")

---

**You're done.** Submit `submission.csv` to Kaggle.

Tips:
- If you're under leaderboard time pressure, set `TTA_PERMUTATIONS=1` for a quick first submission, then re-run with `TTA_PERMUTATIONS=8` for your "final" submission once you're confident.
- Keep `submission_zeroshot.csv` (run this notebook before training) as a sanity reference.
